# 03 Gold star schema

Builds the tables the Power BI semantic model reads with Direct Lake:

| Table | Grain | Notes |
|---|---|---|
| `gold.fact_sales` | one completed order line | revenue, cost and gross profit in DKK; returns are negative |
| `gold.fact_target` | city and month | monthly revenue target |
| `gold.dim_date` | day | calendar plus fiscal year (June to May) |
| `gold.dim_store` | store | channel, city, open or closed |
| `gold.dim_city` | city | links stores and targets |
| `gold.dim_product` | SKU | category, price, cost, status |

Keys are `xxhash64` of the business key, so they stay the same from run to run. The gold
tables are rebuilt in full on every run, which takes seconds at this size.

In [ ]:
run_date = ""

In [ ]:
import datetime as dt

from pyspark.sql import functions as F

In [ ]:
%run nb_lib_retail

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
run_date = run_date or dt.date.today().isoformat()

sales = spark.table("silver.sales_lines")
products = spark.table("silver.products")
stores = spark.table("silver.stores")
targets = spark.table("silver.sales_targets")

first_sale = sales.agg(F.min("sale_date")).first()[0]
last_target = targets.agg(F.max("month_start")).first()[0]
start = dt.date(first_sale.year if first_sale.month >= 6 else first_sale.year - 1, 6, 1)
end = dt.date(last_target.year if last_target.month < 6 else last_target.year + 1, 5, 31)

tables = {
    "dim_date": build_dim_date(spark, start, end),
    "dim_city": build_dim_city(stores),
    "dim_store": build_dim_store(stores, run_date),
    "dim_product": build_dim_product(products, run_date),
    "fact_sales": build_fact_sales(sales, products),
    "fact_target": build_fact_target(targets),
}

for name, df in tables.items():
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"gold.{name}")
    print(f"gold.{name}: {spark.table('gold.' + name).count():,} rows")

In [ ]:
display(spark.sql("""
    SELECT d.fiscal_year, s.channel, round(sum(f.net_revenue_dkk) / 1e6, 2) AS revenue_m_dkk,
           round(sum(f.gross_profit_dkk) / sum(f.net_revenue_dkk), 3) AS gross_margin
    FROM gold.fact_sales f
    JOIN gold.dim_date d ON f.date_key = d.date_key
    JOIN gold.dim_store s ON f.store_key = s.store_key
    GROUP BY d.fiscal_year, s.channel ORDER BY d.fiscal_year, s.channel
"""))